# Chapter 4: Deep Deterministic Policy Gradient
### **RL: The Seminal Papers** by Rahul Shirale

Welcome to the interactive companion notebook for Chapter 4. We implement **DDPG** from Lillicrap et al. (2015), "Continuous Control with Deep Reinforcement Learning." DDPG extends DQN to continuous action spaces using a deterministic actor-critic architecture with soft target updates and Gaussian exploration noise. We verify the full pipeline on `Pendulum-v1`, which reaches a competent policy in under ten minutes on a standard CPU.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rshirale/rl-seminal-papers/blob/main/src/part_2_methods/ch04_ddpg/Chapter4_DDPG.ipynb)

## 1. Setup
The cell below installs dependencies. In Google Colab, uncomment and run it. Locally, use `make install-full` from the repo root.

In [ ]:
# Uncomment in Google Colab. The specifiers are quoted because an
# unquoted ">=" is read by the shell as a redirection.
# %pip install "torch>=2.0.0" "gymnasium[classic-control]>=1.0,<2.0" matplotlib numpy


In [ ]:
import copy
import os
import random
from collections import deque

import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch: {torch.__version__} | Gymnasium: {gym.__version__}")


## 2. Why continuous actions need a different approach

DQN selects actions by computing $Q(s, a)$ for every action and taking the argmax. This works when the action space is small and discrete — CartPole has two actions, Atari games have up to 18. But for a robot arm with 17 joints, each joint torque is a real number in some range $[-t_{max}, t_{max}]$. The argmax over an infinite continuous space is intractable.

DDPG's solution is a **deterministic actor** $\mu(s \mid \theta^\mu)$: a separate neural network that directly outputs the action. The critic $Q(s, a \mid \theta^Q)$ then evaluates that specific action. Training the actor reduces to maximising $Q(s, \mu(s))$ with respect to $\theta^\mu$ via the chain rule — no search over the action space required:

$$\nabla_{\theta^\mu} J \approx \mathbb{E}\left[\nabla_a Q(s, a \mid \theta^Q)\big|_{a=\mu(s)} \cdot \nabla_{\theta^\mu} \mu(s \mid \theta^\mu)\right]$$

This is the **deterministic policy gradient theorem** (Silver et al., 2014) at the heart of DDPG.

## 3. The actor — deterministic policy $\mu(s \mid \theta^\mu)$

Two hidden layers of 400 and 300 units with ReLU activations, matching the paper's specification. The output passes through `tanh` and is scaled by `max_action`, so the policy always outputs actions within the environment's bounds without requiring hard clipping during training.

*Pedagogical Note:* The Lillicrap et al. paper places immense emphasis on uniform initialization for the final layers (e.g., $\mathcal{U}(-3\times 10^{-3}, 3\times 10^{-3})$) to prevent the networks from outputting saturating actions early in training. We implement this in `reset_parameters()` below.

In [ ]:
class Actor(nn.Module):
    """Deterministic policy mu(s | theta^mu) from Lillicrap et al. (2015).

    state -> FC(400) -> ReLU -> FC(300) -> ReLU -> FC(action_dim) -> tanh * max_action
    """

    def __init__(self, state_dim: int, action_dim: int, max_action: float):
        super().__init__()
        self.l1 = nn.Linear(state_dim, 400)
        self.l2 = nn.Linear(400, 300)
        self.l3 = nn.Linear(300, action_dim)
        self.max_action = max_action
        self.reset_parameters()

    def reset_parameters(self):
        """Uniform init on the output layer, so the policy starts unsaturated.

        The hidden layers need no override: the paper initialises them from
        U(-1/sqrt(fan_in), 1/sqrt(fan_in)), which is nn.Linear's default.
        """
        nn.init.uniform_(self.l3.weight, -3e-3, 3e-3)
        nn.init.uniform_(self.l3.bias, -3e-3, 3e-3)

    def forward(self, state: torch.Tensor) -> torch.Tensor:
        x = torch.relu(self.l1(state))
        x = torch.relu(self.l2(x))
        return self.max_action * torch.tanh(self.l3(x))


# Smoke test - Pendulum-v1: state_dim=3, action_dim=1, max_action=2.0
actor_test = Actor(state_dim=3, action_dim=1, max_action=2.0)
out = actor_test(torch.zeros(1, 3))
print(f"Actor output shape: {out.shape}  (batch=1, action_dim=1)")
print(f"Output value (zero state): {out.item():.4f}  - near zero, not saturated")


## 4. The critic — action-value network $Q(s, a \mid \theta^Q)$

The critic estimates the expected return for taking action $a$ in state $s$. A subtle but important architectural decision from the paper: **the action is not fed in at the input layer**. Instead, the state passes through the first hidden layer alone (building a useful state representation), and the action is concatenated with that 400-unit embedding before the second hidden layer.

This gives the first layer time to extract state features before conditioning on the action, which the authors found to improve stability.

In [ ]:
class Critic(nn.Module):
    """Action-value network Q(s, a | theta^Q) from Lillicrap et al. (2015).

    state -> FC(400) -> ReLU -> cat(action) -> FC(300) -> ReLU -> FC(1)
    """

    def __init__(self, state_dim: int, action_dim: int):
        super().__init__()
        self.l1 = nn.Linear(state_dim, 400)
        self.l2 = nn.Linear(400 + action_dim, 300)  # action concatenated here
        self.l3 = nn.Linear(300, 1)
        self.reset_parameters()

    def reset_parameters(self):
        """Uniform init on the output layer, so Q starts near zero."""
        nn.init.uniform_(self.l3.weight, -3e-3, 3e-3)
        nn.init.uniform_(self.l3.bias, -3e-3, 3e-3)

    def forward(self, state: torch.Tensor, action: torch.Tensor) -> torch.Tensor:
        x = torch.relu(self.l1(state))
        x = torch.cat([x, action], dim=1)           # merge after the first layer
        x = torch.relu(self.l2(x))
        return self.l3(x)


# Smoke test
critic_test = Critic(state_dim=3, action_dim=1)
q_val = critic_test(torch.zeros(1, 3), torch.zeros(1, 1))
print(f"Critic output shape: {q_val.shape}  (batch=1, scalar Q-value)")
print(f"Q-value (zero state, zero action): {q_val.item():.4f}")


## 5. Gaussian noise — exploration for a deterministic policy

A deterministic policy returns the same action for the same state, so exploration has to be added from outside the network:

$$a_t^{\text{explore}} = \mu(s_t \mid \theta^\mu) + \mathcal{N}, \qquad \mathcal{N} \sim \mathcal{N}(0, \sigma^2)$$

The original paper used an Ornstein-Uhlenbeck process, whose samples are temporally correlated. Practice since has settled on plain Gaussian noise: it drops two hyperparameters and performs comparably on the standard benchmarks, which is why TD3, Stable-Baselines3 and OpenAI Spinning Up all default to it.

One thing worth being precise about: i.i.d. draws are *independent*, meaning the sign of one sample tells you nothing about the sign of the next. They do not alternate — independence is not anti-correlation.

`sigma = 0.2` matches the scale used in the paper. Annealing is off by default; passing `sigma_final` tapers exploration linearly as the policy matures, which is common in long production runs but is not needed to converge on `Pendulum-v1`.


In [ ]:
class GaussianNoise:
    """I.i.d. Gaussian exploration noise, with optional linear annealing."""

    def __init__(self, size, sigma=0.2, sigma_final=None, decay_steps=100_000):
        if sigma_final is not None and sigma_final > sigma:
            raise ValueError("sigma_final must not exceed sigma")
        self.size = size
        self.sigma_start = sigma
        self.sigma_final = sigma if sigma_final is None else sigma_final
        self.decay_steps = max(1, decay_steps)
        self.steps = 0

    @property
    def sigma(self):
        """Current scale, interpolated linearly toward sigma_final."""
        fraction = min(1.0, self.steps / self.decay_steps)
        return self.sigma_start + fraction * (self.sigma_final - self.sigma_start)

    def reset(self):
        """No-op. Kept so the loop can treat every noise process alike."""

    def sample(self):
        sigma = self.sigma
        self.steps += 1
        return np.random.normal(0.0, sigma, size=self.size)


# Smoke test - constant by default, annealed when asked
constant = GaussianNoise(1)
annealed = GaussianNoise(1, sigma=0.2, sigma_final=0.05, decay_steps=1_000)
for _ in range(1_000):
    annealed.sample()
print(f"Constant sigma after 1000 samples: {constant.sigma:.3f}")
print(f"Annealed sigma after 1000 samples: {annealed.sigma:.3f}")


## 6. Replay buffer

DDPG reuses DQN's experience replay buffer unchanged. Transitions $(s, a, r, s', \text{done})$ are stored in a circular deque of capacity 1,000,000 and sampled uniformly at random to form mini-batches. Sampling breaks the temporal correlation between consecutive transitions, satisfying gradient descent's i.i.d. assumption.

**Architecture Note (RAM vs. VRAM):** We store these millions of transitions on the CPU (system RAM) rather than filling up precious GPU VRAM. Tensors are only pushed to the active `device` (GPU) when a mini-batch is explicitly sampled during training.

In [ ]:
class ReplayBuffer:
    """Circular replay of (s, a, r, s\', done) transitions."""

    def __init__(self, max_size: int = 1_000_000):
        self.buf = deque(maxlen=max_size)

    def push(self, s, a, r, ns, done):
        self.buf.append((s, a, r, ns, done))

    def sample(self, batch_size: int):
        """One uniform mini-batch, as float32 CPU tensors.

        Each field is stacked into a contiguous float32 array first, then
        wrapped with torch.from_numpy, which shares that array's memory rather
        than copying it. Building tensors straight from the list of tuples
        walks Python objects on a path that runs once per environment step.
        """
        batch = random.sample(self.buf, batch_size)
        s, a, r, ns, d = zip(*batch)

        def stack(field):
            return torch.from_numpy(np.asarray(np.stack(field), dtype=np.float32))

        return (
            stack(s),
            stack(a),
            torch.from_numpy(np.asarray(r, dtype=np.float32)).unsqueeze(1),
            stack(ns),
            torch.from_numpy(np.asarray(d, dtype=np.float32)).unsqueeze(1),
        )

    def __len__(self) -> int:
        return len(self.buf)


# Smoke test
buf = ReplayBuffer(max_size=10_000)
for _ in range(500):
    buf.push(np.zeros(3), np.zeros(1), -1.0, np.zeros(3), 0.0)
s, a, r, ns, d = buf.sample(64)
print(f"Buffer size: {len(buf)} | states {tuple(s.shape)}, "
      f"actions {tuple(a.shape)}, rewards {tuple(r.shape)}, dtype {s.dtype}")


## 7. Soft target updates

DQN periodically hard-copies online weights into the target network every $C$ steps. DDPG replaces this with a **soft update** that nudges the target network a small step toward the online network at every training step:

$$\theta' \leftarrow \tau \theta + (1 - \tau) \theta'$$

With $\tau = 0.001$, the target changes by only 0.1% per step, making the learning target extremely stable. This is one of the three key contributions of the paper (alongside replay and the deterministic policy gradient). The paper's ablation study shows that removing target networks causes training to diverge on most tasks.

## 8. The DDPG agent

The `DDPGAgent` class assembles all five components:

- **Online actor** $\mu(s \mid \theta^\mu)$ + **target actor** $\mu'$ (frozen copy, soft-updated)
- **Online critic** $Q(s, a \mid \theta^Q)$ + **target critic** $Q'$ (frozen copy, soft-updated)
- **`GaussianNoise`** for exploration
- **`ReplayBuffer`** for experience storage

The `train()` method runs one mini-batch update:

1. Critic update — minimise the squared Bellman residual, with the target networks supplying a stable regression target
2. Actor update — maximise $Q(s, \mu(s))$, ascending the gradient the critic hands back through the action
3. Soft-update both target networks

One efficiency detail in step 2 is worth reading closely. The actor's loss backpropagates *through* the critic to reach the actor's weights, so autograd would otherwise build and populate a full set of critic gradients that nothing consumes — only `actor_opt.step()` runs. Freezing the critic's parameters for the duration of that backward pass skips the wasted half of the graph. The behaviour is identical; the work is not.


In [ ]:
class DDPGAgent:
    """DDPG from Lillicrap et al. (2015), Algorithm 1.

    Deterministic actor + action-value critic, with soft target updates.
    """

    def __init__(
        self,
        state_dim:   int,
        action_dim:  int,
        max_action:  float,
        gamma:       float = 0.99,
        tau:         float = 0.001,
        actor_lr:    float = 1e-4,
        critic_lr:   float = 1e-3,
        batch_size:  int   = 64,
        buffer_size: int   = 1_000_000,
        sigma:       float = 0.2,
    ):
        self.gamma      = gamma
        self.tau        = tau
        self.batch_size = batch_size
        self.max_action = max_action

        # Online networks
        self.actor  = Actor(state_dim, action_dim, max_action).to(device)
        self.critic = Critic(state_dim, action_dim).to(device)

        # Target networks - exact copies at t=0, thereafter moved only by
        # the soft update. Frozen, so no optimizer ever touches them.
        self.actor_target  = copy.deepcopy(self.actor)
        self.critic_target = copy.deepcopy(self.critic)
        for p in self.actor_target.parameters():
            p.requires_grad = False
        for p in self.critic_target.parameters():
            p.requires_grad = False

        self.actor_opt  = torch.optim.Adam(self.actor.parameters(),  lr=actor_lr)
        self.critic_opt = torch.optim.Adam(self.critic.parameters(), lr=critic_lr)

        self.replay = ReplayBuffer(buffer_size)
        self.noise  = GaussianNoise(action_dim, sigma=sigma)

    def select_action(self, state, explore: bool = True):
        """A clipped action, optionally with exploration noise."""
        s_t = torch.as_tensor(state, dtype=torch.float32, device=device)
        with torch.no_grad():
            action = self.actor(s_t).cpu().numpy()
        if explore:
            action = action + self.noise.sample()
        return np.clip(action, -self.max_action, self.max_action)

    def store(self, s, a, r, ns, done):
        """`done` is the termination flag, never `terminated or truncated`."""
        self.replay.push(s, a, r, ns, done)

    def train(self):
        """One critic step, one actor step, two soft updates.

        Returns (critic_loss, actor_loss), or (None, None) while the buffer
        holds less than one batch.
        """
        if len(self.replay) < self.batch_size:
            return None, None

        s, a, r, ns, d = [t.to(device)
                          for t in self.replay.sample(self.batch_size)]

        # --- Critic update ---
        with torch.no_grad():
            na = self.actor_target(ns)
            nq = self.critic_target(ns, na)
            y  = r + self.gamma * (1.0 - d) * nq

        c_loss = F.mse_loss(self.critic(s, a), y)
        self.critic_opt.zero_grad()
        c_loss.backward()
        self.critic_opt.step()

        # --- Actor update (ascend Q by descending -Q) ---
        # Critic gradients here would be computed and then discarded, so
        # skip building them.
        self._set_critic_requires_grad(False)
        a_loss = -self.critic(s, self.actor(s)).mean()
        self.actor_opt.zero_grad()
        a_loss.backward()
        self.actor_opt.step()
        self._set_critic_requires_grad(True)

        # --- Soft target updates ---
        self._soft_update(self.actor_target,  self.actor)
        self._soft_update(self.critic_target, self.critic)

        return c_loss.item(), a_loss.item()

    def reset_noise(self):
        self.noise.reset()

    def _set_critic_requires_grad(self, flag: bool):
        for p in self.critic.parameters():
            p.requires_grad = flag

    def _soft_update(self, target: nn.Module, source: nn.Module):
        """theta' <- tau * theta + (1 - tau) * theta'."""
        with torch.no_grad():
            for tp, sp in zip(target.parameters(), source.parameters()):
                tp.copy_(self.tau * sp + (1.0 - self.tau) * tp)


print("DDPGAgent defined.")


## 9. Training on Pendulum-v1

`Pendulum-v1` is a classic continuous-control benchmark. The state is a three-element vector $(\cos\theta,\, \sin\theta,\, \dot{\theta})$ and the single continuous action is torque $\in [-2, 2]$. The reward is approximately $-(\theta^2 + 0.1\dot{\theta}^2 + 0.001\tau^2)$, so the goal is to hold the pendulum upright, still, and with minimal effort. A random policy scores about $-1200$; a converged agent scores above $-200$.

Two details in the loop below are easy to get wrong, and both are worth pausing on.

**Warmup uses uniform random actions, not the policy.** A freshly initialised deterministic actor emits nearly the same action in every state — its output layer starts at $\mathcal{U}(-3\times10^{-3}, 3\times10^{-3})$ precisely so that it does. Filling the buffer from that policy would hand the critic a thousand near-identical torques and almost no information about the action space it has to evaluate.

**We store `terminated`, not `terminated or truncated`.** `Pendulum-v1` never terminates; it only truncates at its 200-step time limit. Collapsing the two would zero the bootstrap term on the final transition of *every* episode, teaching the critic that states at $t=200$ are worthless. The value of a state does not depend on how much clock is left — which is exactly why Gymnasium splits the old `done` flag into two.


In [ ]:
# Environment preview and action space information
env_preview = gym.make("Pendulum-v1", render_mode="rgb_array")
env_preview.reset(seed=42)
frame = env_preview.render()
env_preview.close()

env_info = gym.make("Pendulum-v1")
print("Observation space:", env_info.observation_space)
print("Action space:     ", env_info.action_space)
print(f"Max action:        {float(env_info.action_space.high[0]):.1f}")
env_info.close()

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(frame)
ax.axis("off")
ax.set_title("Pendulum-v1 — swing up and balance the pole using continuous torque")
plt.tight_layout()
plt.show()

In [ ]:
from IPython.display import clear_output

# Overridable so the repo's test suite can execute this notebook quickly.
EPISODES     = int(os.environ.get("CH4_NUM_EPISODES", 200))
MAX_STEPS    = 200      # Pendulum-v1's time limit
WARMUP_STEPS = 1_000    # uniform random actions before the policy takes over
PRINT_EVERY  = 10
SEED         = 0

env = gym.make("Pendulum-v1")

# Seed before building the agent, so weight init is covered too.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
env.reset(seed=SEED)
env.action_space.seed(SEED)

agent = DDPGAgent(
    state_dim  = env.observation_space.shape[0],
    action_dim = env.action_space.shape[0],
    max_action = float(env.action_space.high[0]),
)

returns       = []
critic_losses = []
actor_losses  = []
total_steps   = 0

for ep in range(1, EPISODES + 1):
    state, _ = env.reset()
    agent.reset_noise()
    ep_return = 0.0

    for _ in range(MAX_STEPS):
        if total_steps < WARMUP_STEPS:
            action = env.action_space.sample()      # uniform, not the policy
        else:
            action = agent.select_action(state, explore=True)

        next_state, reward, terminated, truncated, _ = env.step(action)

        # `terminated` only - a time-limit cutoff is not a terminal state.
        agent.store(state, action, reward, next_state, float(terminated))

        if total_steps >= WARMUP_STEPS:
            c_loss, a_loss = agent.train()
            if c_loss is not None:
                critic_losses.append(c_loss)
                actor_losses.append(a_loss)

        state       = next_state
        ep_return  += reward
        total_steps += 1

        if terminated or truncated:
            break

    returns.append(ep_return)

    if ep % PRINT_EVERY == 0:
        avg = np.mean(returns[-PRINT_EVERY:])
        clear_output(wait=True)

        window  = 20
        rolling = [
            np.mean(returns[max(0, i - window + 1): i + 1])
            for i in range(len(returns))
        ]

        fig, ax = plt.subplots(figsize=(10, 3))
        ax.plot(returns, alpha=0.2, color="#0077BB", label="Episode return")
        ax.plot(rolling, linewidth=2, color="#D97706",
                label=f"{window}-ep moving avg")
        ax.axhline(-200, linestyle="--", color="#117733", linewidth=1,
                   label="Competent threshold (\u2212200)")
        ax.set_title(
            f"Training \u2014 Episode {ep}/{EPISODES}"
            f"  |  Last-{PRINT_EVERY} avg: {avg:.0f}"
        )
        ax.set_xlabel("Episode")
        ax.set_ylabel("Return")
        ax.legend(loc="lower right")
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

env.close()
print("Training complete.")


## 10. Learning curve and loss diagnostics

The raw episode return is noisy. The 20-episode moving average reveals the underlying trend. Below we also plot the critic and actor losses, which provide a diagnostic view of what the agent is learning:
- **Critic loss** (MSE of the Bellman residual) should generally decrease and stabilise as the value estimate improves.
- **Actor loss** (negative mean Q-value) should trend downward (become more negative) as the policy finds higher-value actions.

In [ ]:
window         = 20
returns_series = np.array(returns)
rolling_avg    = [
    np.mean(returns_series[max(0, i - window + 1): i + 1])
    for i in range(len(returns_series))
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Episode returns
axes[0].plot(returns_series, alpha=0.25, color="#0077BB", label="Episode return")
axes[0].plot(rolling_avg, linewidth=2, color="#D97706",
             label=f"{window}-ep moving avg")
axes[0].axhline(-200, linestyle="--", color="#117733", linewidth=1,
                label="Competent threshold")
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Return")
axes[0].set_title("DDPG on Pendulum-v1")
axes[0].legend(loc="lower right", fontsize=8,
               frameon=True, facecolor="white", framealpha=1.0,
               edgecolor="#cccccc")
axes[0].grid(True, alpha=0.3)


def smooth(values, w=200):
    """Moving average, shrinking the window rather than returning empty."""
    w = max(1, min(w, len(values)))
    return np.convolve(values, np.ones(w) / w, mode="valid")


# Critic loss
axes[1].plot(smooth(critic_losses), linewidth=1.5, color="#CC3311")
axes[1].set_xlabel("Training step")
axes[1].set_ylabel("MSE loss")
axes[1].set_title("Critic loss (Bellman residual)")
axes[1].grid(True, alpha=0.3)

# Actor loss
axes[2].plot(smooth(actor_losses), linewidth=1.5, color="#117733")
axes[2].set_xlabel("Training step")
axes[2].set_ylabel("\u2212Q (actor loss)")
axes[2].set_title("Actor loss (negative mean Q-value)")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

final_window = min(PRINT_EVERY, len(returns_series))
final_avg    = np.mean(returns_series[-final_window:])
print(f"Final {final_window}-episode average: {final_avg:.1f}")
print(f"Competent threshold: \u2212200  |  "
      f"{'PASSED' if final_avg > -200 else 'not yet reached'}")


## 11. Using the module files

In a production setting, import directly from the companion scripts rather than redefining classes in the notebook:

In [ ]:
# The module files carry the same code, plus seeding, CLI flags and the
# ablation switches the notebook leaves out.
#
# From this directory:
#   from actor           import Actor
#   from critic          import Critic
#   from gaussian_noise  import GaussianNoise
#   from parameter_noise import AdaptiveParameterNoise
#   from replay_buffer   import ReplayBuffer
#   from ddpg_agent      import DDPGAgent
#
# From the repo root, as a package:
#   from src.part_2_methods.ch04_ddpg import DDPGAgent
#
# Full training run, reproducible:
#   python -m src.part_2_methods.ch04_ddpg.train_pendulum --seed 0
#
# Component ablation (target networks / soft vs hard updates):
#   python -m src.part_2_methods.ch04_ddpg.ablation --seeds 0 1 2


## 12. Going further — parameter-space noise

Action noise perturbs what the policy *does*. Parameter-space noise (Plappert et al., ICLR 2018) perturbs what the policy *is*: you jitter a copy of the actor's weights once per episode and roll out that perturbed policy.

$$\hat{\theta} = \theta + \mathcal{N}(0, \sigma^2 I), \qquad a_t = \mu(s_t \mid \hat{\theta})$$

The difference that matters is consistency. Action noise gives an agent that twitches — two visits to the same state produce two different actions, so an exploratory choice is never followed through. A weight perturbation is a *different policy*: deterministic, state-dependent, and coherent for a whole episode. That is what gets a manipulator to attempt a genuinely different grasp rather than jittering around the one it already knows.

The catch is that $\sigma$ has no interpretable scale — the same perturbation changes behaviour wildly depending on the network's current weights, and that relationship drifts as training proceeds. The fix is to adapt it: measure how far the perturbed policy's actions actually moved, and scale $\sigma$ to hold that distance at a target you *can* reason about, namely the action-space $\sigma$ you would otherwise have used.

`Pendulum-v1` does not need this — Gaussian action noise solves it comfortably. It is here because it is the first thing to reach for when action noise stalls on a harder task. Note that only the actor is perturbed: applying learnable parameter noise to the critic has been reported to destabilise training.


In [ ]:
def perturb(actor, sigma):
    """A weight-perturbed copy of the actor. The original is untouched."""
    perturbed = copy.deepcopy(actor)
    with torch.no_grad():
        for p in perturbed.parameters():
            p.add_(torch.randn_like(p) * sigma)
    return perturbed


def action_distance(actor, perturbed_actor, states):
    """RMS action difference between two actors on the same states."""
    with torch.no_grad():
        return float(torch.sqrt(torch.mean(
            (actor(states) - perturbed_actor(states)) ** 2)))


# Adapt sigma until the perturbed policy is as different as sigma=0.2 of
# action noise would have made it.
TARGET, COEFFICIENT = 0.2, 1.01
sigma = 0.05
probe_states = torch.as_tensor(
    np.stack([env.observation_space.sample() for _ in range(256)]),
    dtype=torch.float32,
).to(device)

for step in range(200):
    distance = action_distance(agent.actor, perturb(agent.actor, sigma),
                               probe_states)
    sigma = sigma * COEFFICIENT if distance < TARGET else sigma / COEFFICIENT
    if step % 50 == 0:
        print(f"step {step:3d} | sigma {sigma:.4f} | action distance {distance:.4f}")

print(f"\nConverged sigma: {sigma:.4f} (target action distance {TARGET})")
